In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from optuna_integration import OptunaSearchCV
from optuna.distributions import CategoricalDistribution,FloatDistribution
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss
import warnings
from optuna.logging import set_verbosity, WARNING
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
set_verbosity(WARNING)

# 1.Binary Classification Neural Networks

In [3]:
df = pd.read_csv('../0.dataset/dataset_serangan_jantung_clean.csv')
df_x = df.drop(columns='Serangan_Jantung')
df_y = df['Serangan_Jantung']

In [4]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

In [5]:
params = {
    'hidden_layer_sizes': CategoricalDistribution([
        (100,), (50, 50), (100, 50), 
        (10, 10), (20, 20), (10, 10, 10), (30, 20, 10)
    ]),
    'activation': CategoricalDistribution(['relu', 'tanh', 'logistic']),
    'solver': CategoricalDistribution(['adam', 'sgd']),
    'alpha': FloatDistribution(1e-5, 1e-1, log=True),
    'learning_rate_init': FloatDistribution(1e-4, 1e-1, log=True),
    'learning_rate': CategoricalDistribution(['constant', 'invscaling', 'adaptive']),
    'batch_size': CategoricalDistribution([16, 32, 64, 128, 'auto'])
}
mlp = MLPClassifier()
optuna_search = OptunaSearchCV(estimator=mlp,param_distributions=params,n_trials=10,cv=6,
                               scoring='accuracy',random_state=42,n_jobs=-1)
optuna_search.fit(X_train,y_train)
best_model_mlp = optuna_search.best_estimator_

sfs_dct = SequentialFeatureSelector(estimator=best_model_mlp,n_features_to_select=5,direction='backward')
sfs_dct.fit(X_train,y_train)

X_train_selected = sfs_dct.transform(X_train)
X_test_selected = sfs_dct.transform(X_test)

fitur_terpilih = X_train.columns[sfs_dct.get_support()]
best_model_mlp.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {optuna_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {optuna_search.best_score_*100:.2f}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

C:\Users\HP\AppData\Local\Temp\ipykernel_5880\762896999.py:14: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  optuna_search = OptunaSearchCV(estimator=mlp,param_distributions=params,n_trials=10,cv=6,



Hyperparameter Terbaik: {'hidden_layer_sizes': (100,), 'activation': 'relu', 'solver': 'sgd', 'alpha': 0.0036648264821697437, 'learning_rate_init': 0.00043146199067107594, 'learning_rate': 'adaptive', 'batch_size': 16}

Akurasi Validasi Terbaik: 76.92

Fitur Terbaik Yang Terpilih:
['Usia', 'Tipe_Nyeri_Dada', 'Detak_Jantung_Max', 'Angina_Olahraga', 'Oldpeak_ST']


In [6]:
test_accuracy = best_model_mlp.score(X_test_selected,y_test)
train_accruacy = best_model_mlp.score(X_train_selected,y_train)

y_pred_test = best_model_mlp.predict(X_test_selected)
y_prob_test = best_model_mlp.predict_proba(X_test_selected)

y_pred_train = best_model_mlp.predict(X_train_selected)
y_prob_train = best_model_mlp.predict_proba(X_train_selected)

print(f'\nAkurasi Pada Data Test: {test_accuracy*100:.2f}')
print(f'\nAkurasi Pada Data Train: {train_accruacy*100:.2f}')


Akurasi Pada Data Test: 74.00

Akurasi Pada Data Train: 77.00


In [7]:
def evaluate_model(pred,test,prob,evaluate,model_name='Neural Networks'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [8]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [9]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
             Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Neural Networks  Training      0.77   0.779392  0.844063  0.810440  0.837452  0.487579  Good Fit
1  Neural Networks      Test      0.74   0.779661  0.779661  0.779661  0.818015  0.518858  Good Fit




In [10]:
from sklearn.preprocessing import StandardScaler
feauter_numerik = ['Usia','Tekanan_Darah_Rest','Kolesterol','Detak_Jantung_Max','Oldpeak_ST','BMI']
data_5_pasien = {
    'Usia': [63, 45, 56, 38, 50],
    'Jenis_Kelamin': [3, 1, 0, 1, 0],
    'Tipe_Nyeri_Dada': [-0.3, 1.20, 0.08, -1.05, -0.48],
    'Tekanan_Darah_Rest': [140, 130, 122, 110, 120],
    'Kolesterol': [69, 100, 122, 115, 190],
    'Gula_Darah_Puasa': [1, 0, 0, 0, 1],
    'EKG_Rest': [1, 2, 1, 1, 1],
    'Detak_Jantung_Max': [-0.98, 0.90, -1.05, 0.29, -1.58],
    'Angina_Olahraga': [44, 60, 56, 75, 14],
    'Oldpeak_ST': [19, 59, 73, 76, 18],
    'Kemiringan_ST': [1.28, 2.0, 1.5, 1.5, 2.0],
    'Jumlah_Pembuluh_Darah': [46, 60, 4, 0, 3],
    'Thalassemia': [81, 90, 1, 4, 1],
    'BMI': [10.77, 81.24, 24.5, 28.1, 29.3]
}
scaler = StandardScaler()
data_pasien= pd.DataFrame(data_5_pasien)
data_pasien[feauter_numerik] = scaler.fit_transform(data_pasien[feauter_numerik])
target_asli_pasien = [1, 0, 1, 0, 1] 

data_pasien_baru_x = data_pasien[fitur_terpilih]
data_pasien_baru_y = target_asli_pasien
data_pasien_baru_x

,Usia,Tipe_Nyeri_Dada,Detak_Jantung_Max,Angina_Olahraga,Oldpeak_ST
0,1.458427,-0.30,-0.535966,44,-1.173811
1,-0.625040,1.20,1.495518,60,0.391270
2,0.648190,0.08,-0.611606,56,0.939049
3,-1.435277,-1.05,0.836366,75,1.056430
4,-0.046299,-0.48,-1.184312,14,-1.212938


In [11]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
# data_pasien_baru = df_x.sample(2, random_state=10)
data_pasien_baru = data_pasien[fitur_terpilih]
prediksi_pasien = best_model_mlp.predict(data_pasien_baru)
probabilitas_pasien = best_model_mlp.predict_proba(data_pasien_baru)

hasil_df = data_pasien_baru.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_mlp.score(data_pasien_baru_x, data_pasien_baru_y)
prediksi_pasien = best_model_mlp.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_mlp.predict_proba(data_pasien_baru_x)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))

hasil_df[['Peluang Aman(%)','Peluang Terkena (%)','Prediksi Serangan Jantung']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 60.00%

Tabel Skor Evaluasi Lengkap Data Baru:
          Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Neural Networks Data_Baru       0.6        0.6     1.0      0.75 0.333333 14.417461


,Peluang Aman(%),Peluang Terkena (%),Prediksi Serangan Jantung
0,0.000000e+00,100.0,1
1,0.000000e+00,100.0,1
2,0.000000e+00,100.0,1
3,0.000000e+00,100.0,1
4,4.882166e-07,100.0,1
